In [6]:
"""
ClickUp Task Import Script
Imports tasks from CSV to ClickUp with dependencies and custom fields
"""
import os
import time
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta
from typing import Optional, List, Dict, Tuple, Any

# ==================== ClickUp Configuration ====================
SPACE_ID = "90159483029"
TEAM_ID = "90152198197"
TARGET_FOLDER_NAME = "10 PV | Utility-scale"
TARGET_LIST_NAME = "PV0000-00 | TEST v03"
CSV_PATH_SIMPLIFIED = "DEPA_PV_Register_ClickUp_import_v3.csv"

# Optional template configuration (uncomment to use)
# TASK_TEMPLATE_NAME = "HTL | TEMPLATE | Execution"
# TASK_TEMPLATE_ID = "t-86c7t51ff"

CLICKUP_TOKEN = "pk_56660333_WATA0RDNID48ZA30VX30Z2CRSNB16SN3"
DRY_RUN = False
BASE_URL = "https://api.clickup.com/api/v2"

# Column validation and normalization is handled entirely by normalize_column_names()
# which enforces strict naming standards

# ==================== HTTP Session Setup ====================
session = requests.Session()
session.headers.update({
    "Authorization": CLICKUP_TOKEN,
    "Accept": "application/json",
    "Content-Type": "application/json",
})

print("✓ Configuration loaded")

✓ Configuration loaded


In [7]:
"""
Core utility functions for HTTP requests, date conversion, and data parsing
"""

def api_request(
    method: str,
    url: str,
    *,
    params: Optional[Dict] = None,
    json_data: Optional[Dict] = None,
    expected_status: Tuple[int, ...] = (200, 201),
    retries: int = 6
) -> Dict[str, Any]:
    """
    Make HTTP request to ClickUp API with retry logic.
    
    Args:
        method: HTTP method (GET, POST, PUT, etc.)
        url: Full URL to request
        params: Query parameters
        json_data: JSON body data
        expected_status: Tuple of acceptable status codes
        retries: Number of retry attempts for rate limits/server errors
    
    Returns:
        JSON response as dictionary
    
    Raises:
        RuntimeError: If request fails after all retries
    """
    if DRY_RUN:
        print(f"[DRY_RUN] {method} {url} params={params} json={json_data}")
        return {}
    
    last_error = None
    for attempt in range(retries):
        try:
            response = session.request(
                method, url, params=params, json=json_data, timeout=60
            )
            
            if response.status_code in expected_status:
                return response.json() if response.text else {}
            
            # Handle rate limiting and server errors with exponential backoff
            if response.status_code == 429 or response.status_code >= 500:
                wait_time = min(60, 2 ** attempt)
                print(f"  ⏳ Rate limit/server error, waiting {wait_time}s (attempt {attempt+1}/{retries})")
                time.sleep(wait_time)
                last_error = (response.status_code, response.text[:400])
                continue
            
            # Other errors - fail immediately
            error_snippet = (response.text or "")[:400]
            raise RuntimeError(
                f"{method} {url} failed: {response.status_code}\n{error_snippet}"
            )
            
        except requests.exceptions.RequestException as e:
            last_error = str(e)
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise
    
    raise RuntimeError(
        f"{method} {url} failed after {retries} retries. Last error: {last_error}"
    )


def parse_date_to_milliseconds(date_str: str) -> int:
    """
    Parse date string to Unix timestamp in milliseconds (midday UTC).
    
    Supports formats: YYYY-MM-DD, YYYY/MM/DD
    
    Args:
        date_str: Date string to parse
    
    Returns:
        Unix timestamp in milliseconds
    
    Raises:
        ValueError: If date format is not recognized
    """
    date_obj = None
    
    for date_format in ["%Y-%m-%d", "%Y/%m/%d"]:
        try:
            date_obj = datetime.strptime(date_str, date_format)
            break
        except ValueError:
            continue
    
    if date_obj is None:
        raise ValueError(
            f"Cannot parse date '{date_str}'. Expected YYYY-MM-DD or YYYY/MM/DD format."
        )
    
    # Set to midday UTC to avoid timezone issues
    date_obj = date_obj.replace(tzinfo=timezone.utc) + timedelta(hours=12)
    return int(date_obj.timestamp() * 1000)


def days_to_milliseconds(days: float) -> int:
    """Convert days to milliseconds."""
    return int(float(days) * 24 * 60 * 60 * 1000)


def parse_dependencies(dep_cell: Any) -> List[str]:
    """
    Parse dependency cell into list of task names.
    
    Supports separators: semicolon, comma, Chinese semicolon
    
    Args:
        dep_cell: Cell value containing dependencies
    
    Returns:
        List of dependency Task names
    """
    if dep_cell is None or (isinstance(dep_cell, float) and pd.isna(dep_cell)):
        return []
    
    dep_str = str(dep_cell).strip()
    if not dep_str or dep_str.lower() == "nan":
        return []
    
    # Normalize separators and split
    normalized = dep_str.replace("；", ";").replace(",", ";")
    return [chunk.strip() for chunk in normalized.split(";") if chunk.strip()]


def normalize_task_name(name: str) -> str:
    """Normalize task name for fuzzy matching (lowercase, single spaces)."""
    return " ".join(str(name).strip().lower().split())


def is_valid_value(value: Any) -> bool:
    """Check if value is valid (not None, NaN, or empty string)."""
    if value is None:
        return False
    if isinstance(value, float) and pd.isna(value):
        return False
    if str(value).strip().lower() in ["nan", ""]:
        return False
    return True


print("✓ Utility functions defined")

✓ Utility functions defined


In [8]:
"""
Higher-level functions for interacting with ClickUp API
"""

def get_all_templates(team_id: Optional[str] = None) -> pd.DataFrame:
    """
    Fetch and display all ClickUp task templates.
    
    Args:
        team_id: ClickUp Team ID. If None, uses TEAM_ID from config.
    
    Returns:
        DataFrame with template information
    """
    team_id = team_id or TEAM_ID
    
    try:
        response = api_request("GET", f"{BASE_URL}/team/{team_id}/taskTemplate")
        templates = response.get("templates", [])
        
        if not templates:
            print("No templates found.")
            return pd.DataFrame()
        
        template_data = [
            {
                "Template ID": t.get("id", ""),
                "Template Name": t.get("name", ""),
                "Created Date": t.get("date_created", ""),
                "Color": t.get("color", ""),
            }
            for t in templates
        ]
        
        df_templates = pd.DataFrame(template_data)
        print(f"\n{'='*80}")
        print(f"Found {len(df_templates)} ClickUp Templates in Team {team_id}")
        print(f"{'='*80}\n")
        print(df_templates.to_string(index=False))
        print(f"\n{'='*80}\n")
        
        return df_templates
        
    except Exception as e:
        print(f"❌ Error fetching templates: {e}")
        return pd.DataFrame()


def find_folder_by_name(space_id: str, folder_name: str) -> Optional[str]:
    """Find folder ID by name in a space."""
    folders = api_request(
        "GET", f"{BASE_URL}/space/{space_id}/folder",
        params={"archived": "false"}
    ).get("folders", [])
    
    for folder in folders:
        if str(folder.get("name", "")).strip().lower() == folder_name.strip().lower():
            return str(folder.get("id"))
    
    return None


def find_or_create_list(folder_id: str, list_name: str) -> str:
    """Find list by name or create it if it doesn't exist."""
    lists = api_request(
        "GET", f"{BASE_URL}/folder/{folder_id}/list",
        params={"archived": "false"}
    ).get("lists", [])
    
    for lst in lists:
        if str(lst.get("name", "")).strip().lower() == list_name.strip().lower():
            return str(lst.get("id"))
    
    # Create list if not found
    print(f"⚠ List '{list_name}' not found. Creating...")
    created = api_request(
        "POST", f"{BASE_URL}/folder/{folder_id}/list",
        json_data={"name": list_name}
    )
    list_id = str(created.get("id"))
    print(f"✓ Created List: {list_name} (id={list_id})")
    return list_id


def get_custom_fields(list_id: str) -> Tuple[Optional[str], Optional[str], Dict[str, str]]:
    """
    Get custom field IDs for WBS and Branch fields.
    
    Returns:
        Tuple of (wbs_field_id, branch_field_id, branch_options_dict)
    """
    fields = api_request("GET", f"{BASE_URL}/list/{list_id}/field").get("fields", [])
    
    wbs_field_id = None
    branch_field_id = None
    branch_options = {}
    
    for field in fields:
        field_name = str(field.get("name", "")).strip().lower()
        
        if field_name == "wbs":
            wbs_field_id = str(field.get("id"))
            print(f"✓ Found WBS custom field (id={wbs_field_id})")
        
        elif field_name == "branch":
            branch_field_id = str(field.get("id"))
            options = field.get("type_config", {}).get("options", [])
            branch_options = {
                str(opt.get("name", "")).strip().lower(): opt.get("id")
                for opt in options if opt.get("id")
            }
            print(f"✓ Found Branch custom field (id={branch_field_id}) with {len(branch_options)} options")
    
    return wbs_field_id, branch_field_id, branch_options


def find_template_by_name(team_id: str, template_name: str, fallback_id: Optional[str] = None) -> Tuple[Optional[str], bool]:
    """
    Find template ID by name with fallback support.
    
    Returns:
        Tuple of (template_id, use_template)
    """
    if not template_name and not fallback_id:
        return None, False
    
    try:
        response = api_request("GET", f"{BASE_URL}/team/{team_id}/taskTemplate")
        templates = response.get("templates", [])
        
        print(f"\n🔍 Searching for template: '{template_name}'")
        print(f"Found {len(templates)} templates. Checking...")
        
        for template in templates:
            t_name = str(template.get("name", "")).strip()
            t_id = str(template.get("id", ""))
            
            if t_name.lower() == template_name.strip().lower():
                print(f"  ✓ MATCH FOUND: '{t_name}' (id={t_id})")
                return t_id, True
        
        if fallback_id:
            print(f"\n⚠ Template '{template_name}' not found, using fallback: {fallback_id}")
            return fallback_id, True
        
        print(f"\n⚠ Template '{template_name}' not found")
        return None, False
        
    except Exception as e:
        print(f"⚠ Could not search templates: {e}")
        if fallback_id:
            print(f"Using fallback: {fallback_id}")
            return fallback_id, True
        return None, False


def get_or_create_custom_field(
    list_id: str,
    field_name: str,
    field_type: str
) -> str:
    """
    Get custom field ID by name or create it if it doesn't exist.
    If field exists but has wrong type, delete and recreate it.
    
    Args:
        list_id: ClickUp List ID
        field_name: Name of the custom field
        field_type: Type of field (text, number, dropdown, date, etc.)
    
    Returns:
        Custom field ID
    """
    field_type_mapping = {
        "text": "text",
        "number": "number",
        "dropdown": "drop_down",  # ClickUp API uses "drop_down"
        "date": "date",
        "checkbox": "checkbox",
        "email": "email",
        "url": "url",
        "phone": "phone",
        "currency": "currency",
        "percent": "percent",
        "rating": "rating",
        "formula": "formula",
    }
    
    # Normalize field_type by removing underscores and spaces, then converting to lowercase
    normalized_type = field_type.lower().replace("_", "").replace(" ", "")
    mapped_type = field_type_mapping.get(normalized_type, "text")
    
    print(f"  → Requested: '{field_name}' type '{field_type}' → API type: '{mapped_type}'")
    
    # Get existing fields
    fields = api_request("GET", f"{BASE_URL}/list/{list_id}/field").get("fields", [])
    
    for field in fields:
        if str(field.get("name", "")).strip().lower() == field_name.strip().lower():
            existing_type = field.get("type", "")
            field_id = str(field.get("id"))
            
            if existing_type == mapped_type:
                print(f"  ✓ Found existing custom field '{field_name}' with correct type '{existing_type}' (id={field_id})")
                return field_id
            else:
                print(f"  ⚠ Found existing custom field '{field_name}' but type is '{existing_type}' (expected '{mapped_type}')")
                print(f"  → Deleting incorrect field (id={field_id})...")
                
                try:
                    api_request(
                        "DELETE",
                        f"{BASE_URL}/field/{field_id}",
                        expected_status=(200, 204)
                    )
                    print(f"  ✓ Deleted field '{field_name}'")
                except Exception as e:
                    print(f"  ⚠ Could not delete field: {e}")
                    raise RuntimeError(f"Field '{field_name}' exists with wrong type and cannot be deleted. Please delete it manually in ClickUp.")
                
                break
    
    # Create field if not found or was deleted
    print(f"  ⚠ Creating custom field '{field_name}' with type '{mapped_type}'...")
    
    # Build request payload
    payload = {
        "name": field_name,
        "type": mapped_type
    }
    
    # Dropdown fields require type_config with options
    if mapped_type == "drop_down":
        payload["type_config"] = {
            "default": 0,
            "placeholder": None,
            "options": [
                {"name": "Option 1", "color": None, "orderindex": 0},
                {"name": "Option 2", "color": None, "orderindex": 1}
            ]
        }
    
    response = api_request(
        "POST",
        f"{BASE_URL}/list/{list_id}/field",
        json_data=payload
    )
    
    field_id = str(response.get("id") or response.get("field", {}).get("id") or "")
    if not field_id:
        raise RuntimeError(f"Could not create custom field '{field_name}'. Response: {response}")
    
    # Verify the created field type
    created_type = response.get("type") or response.get("field", {}).get("type", "")
    print(f"  ✓ Created custom field '{field_name}' (id={field_id}, type={created_type})")
    
    if created_type != mapped_type:
        print(f"  ⚠⚠⚠ WARNING: Created field has type '{created_type}' but expected '{mapped_type}'!")
    
    return field_id


def get_or_create_checklist(
    task_id: str,
    checklist_name: str
) -> str:
    """
    Get checklist ID by name or create it if it doesn't exist.
    
    Args:
        task_id: ClickUp Task ID
        checklist_name: Name of the checklist
    
    Returns:
        Checklist ID
    """
    # Get existing checklists
    response = api_request("GET", f"{BASE_URL}/task/{task_id}")
    checklists = response.get("checklists", [])
    
    for checklist in checklists:
        if str(checklist.get("name", "")).strip().lower() == checklist_name.strip().lower():
            print(f"    ✓ Found existing checklist '{checklist_name}' (id={checklist.get('id')})")
            return str(checklist.get("id"))
    
    # Create checklist if not found
    print(f"    ⚠ Checklist '{checklist_name}' not found. Creating...")
    
    response = api_request(
        "POST",
        f"{BASE_URL}/task/{task_id}/checklist",
        json_data={"name": checklist_name}
    )
    
    checklist_id = str(response.get("id") or response.get("checklist", {}).get("id") or "")
    if not checklist_id:
        raise RuntimeError(f"Could not create checklist '{checklist_name}'. Response: {response}")
    
    print(f"    ✓ Created checklist '{checklist_name}' (id={checklist_id})")
    return checklist_id


def delete_checklist_item(checklist_id: str, item_id: str) -> None:
    """Delete a checklist item by ID."""
    api_request(
        "DELETE",
        f"{BASE_URL}/checklist/{checklist_id}/checklist_item/{item_id}",
        expected_status=(200, 204)
    )


def add_task_comment(task_id: str, comment_text: str) -> bool:
    """
    Add a comment to a task.
    
    Args:
        task_id: ClickUp Task ID
        comment_text: Comment text
    
    Returns:
        True if successful, False otherwise
    """
    try:
        api_request(
            "POST",
            f"{BASE_URL}/task/{task_id}/comment",
            json_data={"comment_text": comment_text}
        )
        return True
    except Exception as e:
        print(f"    ⚠ Could not add comment to task: {e}")
        return False


def get_task_details(task_id: str) -> Dict[str, Any]:
    """Fetch full task details."""
    return api_request("GET", f"{BASE_URL}/task/{task_id}")


def normalize_comment_text(text: Any) -> str:
    """Normalize comment text for comparison."""
    return str(text).strip()


def get_latest_comment_text(task_id: str) -> Optional[str]:
    """Fetch the most recent comment text for a task."""
    try:
        response = api_request("GET", f"{BASE_URL}/task/{task_id}/comment")
        comments = response.get("comments", []) if isinstance(response, dict) else []
        if not comments:
            return None
        def get_comment_ts(comment: Dict[str, Any]) -> int:
            return int(comment.get("date", comment.get("date_created", 0)) or 0)
        latest = max(comments, key=get_comment_ts)
        comment_text = latest.get("comment_text") or latest.get("text_content") or latest.get("comment") or ""
        normalized = normalize_comment_text(comment_text)
        return normalized if normalized else None
    except Exception as e:
        print(f"    ⚠ Could not fetch latest comment: {e}")
        return None


def get_tasks_in_list_by_name(list_id: str) -> Dict[str, str]:
    """Fetch all tasks in a list and return mapping of normalized name -> task_id."""
    tasks_by_name: Dict[str, str] = {}
    page = 0
    while True:
        response = api_request(
            "GET",
            f"{BASE_URL}/list/{list_id}/task",
            params={"archived": "false", "page": page, "include_closed": "true"}
        )
        tasks = response.get("tasks", [])
        if not tasks:
            break
        for task in tasks:
            name = str(task.get("name", "")).strip()
            task_id = str(task.get("id", ""))
            if name and task_id:
                tasks_by_name[normalize_task_name(name)] = task_id
        last_page = response.get("last_page")
        if last_page is True or last_page == page or len(tasks) < 100:
            break
        page += 1
    return tasks_by_name


def get_custom_field_value_map(task_details: Dict[str, Any]) -> Dict[str, Any]:
    """Return mapping of custom field id -> value from task details."""
    fields = task_details.get("custom_fields", [])
    return {str(field.get("id")): field.get("value") for field in fields}


def normalize_field_value(value: Any) -> Any:
    """Normalize field values for comparison."""
    if value is None:
        return None
    if isinstance(value, dict):
        if "value" in value:
            value = value.get("value")
        elif "id" in value:
            value = value.get("id")
    if isinstance(value, list):
        return tuple(sorted(str(v).strip().lower() for v in value if v is not None))
    return str(value).strip().lower()


def normalize_values_list(values: List[str]) -> Any:
    """Normalize CSV values for comparison."""
    if not values:
        return None
    if len(values) == 1:
        return str(values[0]).strip().lower()
    return tuple(sorted(str(v).strip().lower() for v in values))


def should_update_custom_field(
    field_id: str,
    desired_values: List[str],
    current_value_map: Dict[str, Any]
) -> bool:
    """Determine if a custom field value should be updated."""
    desired_norm = normalize_values_list(desired_values)
    current_norm = normalize_field_value(current_value_map.get(field_id))
    if desired_norm is None:
        return False
    return desired_norm != current_norm


def get_checklist_by_name(task_details: Dict[str, Any], checklist_name: str) -> Optional[Dict[str, Any]]:
    """Find checklist details by name in a task."""
    for checklist in task_details.get("checklists", []):
        if normalize_task_name(str(checklist.get("name", ""))) == normalize_task_name(checklist_name):
            return checklist
    return None


def sync_checklist_items(
    task_id: str,
    task_details: Dict[str, Any],
    checklist_name: str,
    desired_items: List[str]
    ) -> Tuple[int, int, int]:
    """
    Sync checklist items to match desired list.
    Returns (created_checklists, added_items, deleted_items).
    """
    created_checklists = 0
    added_items = 0
    deleted_items = 0
    if not desired_items:
        return created_checklists, added_items, deleted_items
    checklist = get_checklist_by_name(task_details, checklist_name)
    if checklist:
        checklist_id = str(checklist.get("id"))
        existing_items = checklist.get("items", [])
    else:
        checklist_id = get_or_create_checklist(task_id, checklist_name)
        created_checklists = 1
        existing_items = []
    existing_map = {
        normalize_task_name(str(item.get("name", ""))): item
        for item in existing_items
        if item.get("name")
    }
    desired_map = {normalize_task_name(item): item for item in desired_items if item}
    # Delete items not in desired
    for norm_name, item in existing_map.items():
        if norm_name not in desired_map:
            item_id = str(item.get("id"))
            if item_id:
                delete_checklist_item(checklist_id, item_id)
                deleted_items += 1
    # Add missing items
    for norm_name, original_name in desired_map.items():
        if norm_name not in existing_map:
            api_request(
                "POST",
                f"{BASE_URL}/checklist/{checklist_id}/checklist_item",
                json_data={"name": original_name},
                expected_status=(200, 201)
            )
            added_items += 1
    return created_checklists, added_items, deleted_items


def parse_custom_field_spec(field_spec: str) -> Tuple[str, str]:
    """
    Parse custom field specification string.
    
    Format: "Custom Field [type] [name]"
    Example: "Custom Field [text] [Department]" -> ("Department", "text")
    
    Args:
        field_spec: Field specification string
    
    Returns:
        Tuple of (field_name, field_type)
    
    Raises:
        ValueError: If format is invalid
    """
    spec = str(field_spec).strip()
    if not spec.startswith("Custom Field"):
        raise ValueError(f"Invalid custom field spec: {spec}")
    
    # Remove "Custom Field" prefix
    spec = spec[len("Custom Field"):].strip()
    
    # Extract type and name from "[type] [name]"
    import re
    match = re.match(r'\[(.*?)\]\s*\[(.*?)\]', spec)
    if not match:
        raise ValueError(f"Invalid custom field spec format: {spec}")
    
    field_type = match.group(1).strip()
    field_name = match.group(2).strip()
    
    return field_name, field_type


def parse_checklist_spec(checklist_spec: str) -> str:
    """
    Parse checklist specification string.
    
    Format: "Checklist [Name]"
    Example: "Checklist [QA Checklist]" -> "QA Checklist"
    
    Args:
        checklist_spec: Checklist specification string
    
    Returns:
        Checklist name
    
    Raises:
        ValueError: If format is invalid
    """
    spec = str(checklist_spec).strip()
    if not spec.startswith("Checklist"):
        raise ValueError(f"Invalid checklist spec: {spec}")
    
    # Remove "Checklist" prefix
    spec = spec[len("Checklist"):].strip()
    
    # Extract name from "[name]"
    import re
    match = re.match(r'\[(.*?)\]', spec)
    if not match:
        raise ValueError(f"Invalid checklist spec format: {spec}")
    
    return match.group(1).strip()


def create_tasks_from_csv_advanced(
    csv_path: str,
    space_id: str,
    team_id: str,
    folder_name: str,
    list_name: str,
    use_template: bool = False,
    template_id: Optional[str] = None
) -> Dict[str, Any]:
    """
    Create tasks in ClickUp from CSV with advanced field support.
    
    CSV columns:
    - Task name (required)
    - Description (optional)
    - Time estimate (optional, in days)
    - Start date (optional, YYYY-MM-DD format)
    - Due date (optional, YYYY-MM-DD format)
    - Dependencies (optional, semicolon-separated task names)
    - Comments (optional, semicolon-separated comment texts)
    - Custom Field [type] [Name] (optional, semicolon-separated values)
    - Checklist [Name] (optional, semicolon-separated checklist names)
    - Tags (optional, semicolon-separated tag names)
    
    Args:
        csv_path: Path to CSV file
        space_id: ClickUp Space ID
        team_id: ClickUp Team ID
        folder_name: ClickUp Folder name
        list_name: ClickUp List name
        use_template: Whether to use a task template
        template_id: Template ID if use_template is True
    
    Returns:
        Dictionary with statistics and results
    """
    print(f"\n{'='*80}")
    print(f"ADVANCED TASK CREATION FROM CSV")
    print(f"{'='*80}\n")
    
    # Load CSV
    df = load_and_validate_csv(csv_path)
    df = sort_tasks_by_date(df)
    
    # Find folder and list
    folder_id = find_folder_by_name(space_id, folder_name)
    if not folder_id:
        raise RuntimeError(f"Folder not found: {folder_name}")
    print(f"✓ Found Folder: {folder_name} (id={folder_id})")
    
    list_id = find_or_create_list(folder_id, list_name)
    print(f"✓ Found/Created List: {list_name} (id={list_id})\n")
    
    # Parse custom fields and checklists from column names
    custom_field_specs = []
    checklist_specs = []
    
    for col in df.columns:
        if col.startswith("Custom Field"):
            try:
                field_name, field_type = parse_custom_field_spec(col)
                custom_field_specs.append((field_name, field_type, col))
                print(f"✓ Found custom field column: {field_name} ({field_type})")
            except ValueError as e:
                print(f"⚠ Could not parse custom field column '{col}': {e}")
        
        elif col.startswith("Checklist"):
            try:
                checklist_name = parse_checklist_spec(col)
                checklist_specs.append((checklist_name, col))
                print(f"✓ Found checklist column: {checklist_name}")
            except ValueError as e:
                print(f"⚠ Could not parse checklist column '{col}': {e}")
    
    # Ensure custom fields and checklists exist in list
    print(f"\n{'='*80}")
    print(f"CHECKING/CREATING CUSTOM FIELDS AND CHECKLISTS")
    print(f"{'='*80}\n")
    
    custom_field_ids = {}
    for field_name, field_type, col in custom_field_specs:
        field_id = get_or_create_custom_field(list_id, field_name, field_type)
        custom_field_ids[col] = field_id
    
    print()
    
    # Track results
    stats = {
        "created_tasks": 0,
        "existing_tasks": 0,
        "updated_tasks": 0,
        "created_custom_fields": 0,
        "created_checklists": 0,
        "added_comments": 0,
        "added_checklist_items": 0,
        "deleted_checklist_items": 0,
        "failed_tasks": [],
    }
    
    task_name_to_id = {}
    existing_tasks_by_name = get_tasks_in_list_by_name(list_id)
    
    # Create or update tasks
    print(f"{'='*80}")
    print(f"CREATING/UPDATING TASKS")
    print(f"{'='*80}\n")
    
    for idx, (_, row) in enumerate(df.iterrows(), 1):
        task_name = row["Task name"]
        normalized_name = normalize_task_name(task_name)
        existing_task_id = existing_tasks_by_name.get(normalized_name)
        task_details: Dict[str, Any] = {}
        
        try:
            # Create or reuse task
            if existing_task_id:
                task_id = existing_task_id
                stats["existing_tasks"] += 1
                task_details = get_task_details(task_id)
            else:
                if use_template and template_id:
                    response = api_request(
                        "POST",
                        f"{BASE_URL}/list/{list_id}/taskTemplate/{template_id}",
                        json_data={"name": task_name},
                        expected_status=(200, 201)
                    )
                else:
                    response = api_request(
                        "POST",
                        f"{BASE_URL}/list/{list_id}/task",
                        json_data={"name": task_name},
                        expected_status=(200, 201)
                    )
                
                task_id = str(response.get("id") or response.get("task", {}).get("id") or "")
                if not task_id:
                    raise RuntimeError(f"Could not parse task ID from response: {response}")
                stats["created_tasks"] += 1
            
            task_name_to_id[task_name] = task_id
            
            # Update basic fields
            payload = {}
            
            if is_valid_value(row.get("Description")):
                payload["description"] = str(row["Description"])
            
            if is_valid_value(row.get("Start date")):
                payload["start_date"] = parse_date_to_milliseconds(str(row["Start date"]))
                payload["start_date_time"] = False
            
            if is_valid_value(row.get("Due date")):
                payload["due_date"] = parse_date_to_milliseconds(str(row["Due date"]))
                payload["due_date_time"] = False
            
            est_time = row.get("Time estimate")
            if is_valid_value(est_time):
                try:
                    payload["time_estimate"] = days_to_milliseconds(float(est_time))
                except (ValueError, TypeError):
                    pass
            
            if payload:
                api_request("PUT", f"{BASE_URL}/task/{task_id}", json_data=payload, expected_status=(200,))
                stats["updated_tasks"] += 1
            
            # Add tags via separate endpoint
            if is_valid_value(row.get("Tags")):
                tags = [t.strip() for t in str(row["Tags"]).split(";") if t.strip()]
                for tag in tags:
                    try:
                        api_request(
                            "POST",
                            f"{BASE_URL}/task/{task_id}/tag/{tag}",
                            expected_status=(200, 201)
                        )
                    except Exception as e:
                        print(f"    ⚠ Could not add tag '{tag}' to task: {e}")
            
            # Set custom field values (skip dropdown fields as they require option IDs)
            for col, field_id in custom_field_ids.items():
                if is_valid_value(row.get(col)):
                    # Extract field type from column name: "Custom Field [type] [name]"
                    import re
                    field_type_match = re.match(r'Custom Field \[(.*?)\]', col)
                    field_type = field_type_match.group(1).lower() if field_type_match else ""
                    
                    # Skip dropdown fields - they require option IDs which we don't have
                    if field_type == "dropdown":
                        continue
                    
                    values = [v.strip() for v in str(row[col]).split(";") if v.strip()]
                    for value in values:
                        try:
                            api_request(
                                "POST",
                                f"{BASE_URL}/task/{task_id}/field/{field_id}",
                                json_data={"value": value},
                                expected_status=(200,)
                            )
                        except Exception as e:
                            print(f"    ⚠ Could not set custom field value: {e}")
            
            # Add comments (only if different from most recent)
            if is_valid_value(row.get("Comments")):
                comments = [c.strip() for c in str(row["Comments"]).split(";") if c.strip()]
                latest_comment = get_latest_comment_text(task_id) if existing_task_id else None
                for comment in comments:
                    normalized_comment = normalize_comment_text(comment)
                    if not normalized_comment:
                        continue
                    if latest_comment is None or normalized_comment != latest_comment:
                        if add_task_comment(task_id, comment):
                            stats["added_comments"] += 1
                            latest_comment = normalized_comment
            
            # Create and populate checklists
            for checklist_name, col in checklist_specs:
                if is_valid_value(row.get(col)):
                    items = [item.strip() for item in str(row[col]).split(";") if item.strip()]
                    if not task_details:
                        try:
                            checklist_id = get_or_create_checklist(task_id, checklist_name)
                            stats["created_checklists"] += 1
                            for item in items:
                                api_request(
                                    "POST",
                                    f"{BASE_URL}/checklist/{checklist_id}/checklist_item",
                                    json_data={"name": item},
                                    expected_status=(200, 201)
                                )
                        except Exception as e:
                            print(f"    ⚠ Could not create/populate checklist '{checklist_name}': {e}")
                    else:
                        try:
                            created_checklists, added_items, deleted_items = sync_checklist_items(
                                task_id, task_details, checklist_name, items
                            )
                            stats["created_checklists"] += created_checklists
                            stats["added_checklist_items"] += added_items
                            stats["deleted_checklist_items"] += deleted_items
                        except Exception as e:
                            print(f"    ⚠ Could not sync checklist '{checklist_name}': {e}")
            
            action = "Updated" if existing_task_id else "Created"
            print(f"  [{idx}/{len(df)}] ✓ {action}: {task_name} (id={task_id})")
            
        except Exception as e:
            print(f"  [{idx}/{len(df)}] ❌ Failed to process task '{task_name}': {e}")
            stats["failed_tasks"].append({"name": task_name, "error": str(e)})
    
    # Link dependencies if needed
    if "Dependencies" in df.columns or "dependencies" in df.columns:
        print(f"\n{'='*80}")
        print(f"LINKING DEPENDENCIES")
        print(f"{'='*80}\n")
        
        for _, row in df.iterrows():
            child_name = row["Task name"]
            child_id = task_name_to_id.get(child_name)
            
            if not child_id or not is_valid_value(row.get("Dependencies")):
                continue
            
            dependencies = parse_dependencies(row.get("Dependencies"))
            
            for parent_name in dependencies:
                parent_id = task_name_to_id.get(parent_name)
                if not parent_id:
                    # Try normalized match
                    normalized = normalize_task_name(parent_name)
                    for orig_name, orig_id in task_name_to_id.items():
                        if normalize_task_name(orig_name) == normalized:
                            parent_id = orig_id
                            break
                
                if parent_id:
                    try:
                        api_request(
                            "POST",
                            f"{BASE_URL}/task/{child_id}/dependency",
                            json_data={"depends_on": parent_id},
                            expected_status=(200, 201)
                        )
                    except RuntimeError as e:
                        if not ("already" in str(e).lower() or "exist" in str(e).lower()):
                            raise
    
    # Print summary
    print(f"\n{'='*80}")
    print(f"ADVANCED TASK CREATION COMPLETE!")
    print(f"{'='*80}")
    print(f"✓ Created tasks: {stats['created_tasks']}")
    print(f"✓ Existing tasks: {stats['existing_tasks']}")
    print(f"✓ Updated tasks: {stats['updated_tasks']}")
    print(f"✓ Added comments: {stats['added_comments']}")
    print(f"✓ Created checklists: {stats['created_checklists']}")
    print(f"✓ Added checklist items: {stats['added_checklist_items']}")
    print(f"✓ Deleted checklist items: {stats['deleted_checklist_items']}")
    if stats["failed_tasks"]:
        print(f"❌ Failed tasks: {len(stats['failed_tasks'])}")
        for failed in stats["failed_tasks"]:
            print(f"   - {failed['name']}: {failed['error']}")
    print(f"{'='*80}\n")
    
    return stats


print("✓ ClickUp API helper functions and advanced task creation defined")

✓ ClickUp API helper functions and advanced task creation defined


In [9]:
"""
CSV data loading and validation functions
"""

def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize and validate column names according to strict standards.
    
    Valid column patterns:
    - Required: "Task name"
    - Standard optional: "Description", "Tags", "Time estimate", "Start date", "Due date", 
                        "Dependencies", "Comments"
    - Checklists: "Checklist [name]"
    - Custom Fields: "Custom Field [type] [name]"
    
    Normalization rules:
    - Strip leading/trailing whitespace
    - Replace underscores with spaces
    - Collapse multiple spaces to single space
    - Convert to lowercase (except content in brackets)
    
    Args:
        df: DataFrame with potentially non-standard column names
    
    Returns:
        DataFrame with normalized column names
    
    Raises:
        ValueError: If column names don't conform to valid patterns
    """
    import re
    
    df = df.copy()
    
    # Valid standard columns (stored in lowercase for comparison, but output capitalized)
    STANDARD_COLUMNS_LOWER = {
        "task name", "tags", "time estimate", "start date", 
        "due date", "dependencies", "comments", "description"
    }
    
    # Mapping from lowercase to capitalized format
    STANDARD_COLUMNS_CAPITALIZED = {
        "task name": "Task name",
        "tags": "Tags",
        "time estimate": "Time estimate",
        "start date": "Start date",
        "due date": "Due date",
        "dependencies": "Dependencies",
        "comments": "Comments",
        "description": "Description"
    }
    
    normalized_columns = {}
    validation_errors = []
    
    for col in df.columns:
        # Strip and normalize whitespace
        normalized = str(col).strip()
        normalized = normalized.replace("_", " ")
        normalized = re.sub(r' +', ' ', normalized)
        
        # Check if it's a standard column
        if normalized.lower() in STANDARD_COLUMNS_LOWER:
            normalized = STANDARD_COLUMNS_CAPITALIZED[normalized.lower()]
            normalized_columns[col] = normalized
            continue
        
        # Check if it's a Checklist pattern: "Checklist [name]"
        checklist_match = re.match(r'^checklist\s+\[(.*?)\]$', normalized, re.IGNORECASE)
        if checklist_match:
            checklist_name = checklist_match.group(1).strip()
            normalized = f"Checklist [{checklist_name}]"
            normalized_columns[col] = normalized
            continue
        
        # Check if it's a Custom Field pattern: "Custom Field [type] [name]"
        custom_field_match = re.match(
            r'^custom\s+field\s+\[(.*?)\]\s+\[(.*?)\]$', 
            normalized, 
            re.IGNORECASE
        )
        if custom_field_match:
            field_type = custom_field_match.group(1).strip()
            field_name = custom_field_match.group(2).strip()
            normalized = f"Custom Field [{field_type}] [{field_name}]"
            normalized_columns[col] = normalized
            continue
        
        # If none of the patterns match, it's invalid
        validation_errors.append(
            f"Column '{col}' does not conform to valid patterns. "
            f"Must be: a standard column (Task name, Description, Tags, Time estimate, Start date, "
            f"Due date, Dependencies, Comments), Checklist [name], or "
            f"Custom Field [type] [name]"
        )
    
    if validation_errors:
        raise ValueError(
            "Invalid column names found:\n" + "\n".join(validation_errors)
        )
    
    df.rename(columns=normalized_columns, inplace=True)
    
    # Validate that required "Task name" column exists
    if "Task name" not in df.columns:
        raise ValueError(
            f"CSV missing REQUIRED column 'Task name'. Found columns: {list(df.columns)}"
        )
    
    return df


def load_and_validate_csv(csv_path: str) -> pd.DataFrame:
    """
    Load CSV and validate/normalize columns.
    
    Column names are validated strictly according to patterns:
    - Standard columns: Task name (required), Description, Tags, Time estimate, 
                       Start date, Due date, Dependencies, Comments
    - Custom fields: Custom Field [type] [name]
    - Checklists: Checklist [name]
    
    Args:
        csv_path: Path to CSV file
    
    Returns:
        DataFrame with validated and normalized columns
    
    Raises:
        ValueError: If required "Task name" column is missing or invalid column names exist
    """
    # Load CSV file
    df = pd.read_csv(csv_path)
    df = df.dropna(how="all")
    before_rows = len(df)
    
    # Normalize and validate column names to standard format
    df = normalize_column_names(df)
    
    # Drop rows missing required Task name
    df = df[df["Task name"].apply(is_valid_value)].copy()
    dropped = before_rows - len(df)
    if dropped > 0:
        print(f"⚠ Dropped {dropped} empty rows (missing Task name)")
    
    # Clean string columns for standard columns that exist
    df = df.copy()
    for col in ["Task name", "Description", "Tags"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    
    # Ensure dependencies are strings
    if "Dependencies" in df.columns:
        df["Dependencies"] = df["Dependencies"].astype(str)
    
    print(f"✓ Loaded CSV: {csv_path}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Rows: {len(df)}")
    
    return df


def sort_tasks_by_date(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sort tasks by start date and Task name.
    
    Args:
        df: DataFrame with tasks to sort
    
    Returns:
        DataFrame sorted by Start date and Task name
    """
    if "Start date" not in df.columns:
        return df.sort_values(by=["Task name"]).reset_index(drop=True)
    
    def safe_date_parse(s):
        """Parse date string, returning max datetime for invalid dates."""
        try:
            return datetime.strptime(s, "%Y-%m-%d")
        except:
            return datetime(2100, 1, 1)  # Push invalid dates to end
    
    return df.sort_values(
        by=["Start date", "Task name"],
        key=lambda col: col.map(safe_date_parse) if col.name == "Start date" else col
    ).reset_index(drop=True)


print("✓ CSV loading functions defined")

✓ CSV loading functions defined


In [10]:
stats = create_tasks_from_csv_advanced(
    csv_path="07_companies_clickup_import_customfields.csv",
    space_id=SPACE_ID,
    team_id=TEAM_ID,
    folder_name="00 COMMON",
    list_name="COMPANY MASTER",
    use_template=False,
)


ADVANCED TASK CREATION FROM CSV

✓ Loaded CSV: 07_companies_clickup_import_customfields.csv
  Columns: ['Task name', 'Description', 'Custom Field [text] [company name]', 'Custom Field [text] [VAT]', 'Custom Field [text] [company type]', 'Custom Field [text] [name code]']
  Rows: 70
✓ Found Folder: 00 COMMON (id=901514013630)
✓ Found/Created List: COMPANY MASTER (id=901520849191)

✓ Found custom field column: company name (text)
✓ Found custom field column: VAT (text)
✓ Found custom field column: company type (text)
✓ Found custom field column: name code (text)

CHECKING/CREATING CUSTOM FIELDS AND CHECKLISTS

  → Requested: 'company name' type 'text' → API type: 'text'
  ⚠ Creating custom field 'company name' with type 'text'...
  ✓ Created custom field 'company name' (id=5be8785a-f90e-45d5-9b0b-67b90fb83da2, type=text)
  → Requested: 'VAT' type 'text' → API type: 'text'
  ⚠ Creating custom field 'VAT' with type 'text'...
  ✓ Created custom field 'VAT' (id=e7e76358-377b-4424-9874-d13a5

In [11]:
# df = pd.read_csv("03_clickup_pv_worst_case_permitting_import_v6_improvements.csv")
# df["Tags"] = df["Tags"].astype(str).str.replace(",", ";")
# df.to_csv("03_clickup_pv_worst_case_permitting_import_v6_improvements.csv", index=False)